In [19]:
# Import dependencies
import copy
import random
from pathlib import Path

import numpy as np
import pandas as pd
pd.set_option('display.max_columns', None)
import torch
import torch.nn as nn
from lightgbm import LGBMClassifier
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import StandardScaler
from torch.utils.data import DataLoader
from torch.utils.data import TensorDataset

# Set random seeds
def set_seed(seed):
    # Seed all random generators
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    # Seed CUDA when available
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

target_column = "addicted_label"
competition_path = Path(
    "/kaggle/input/competitions/playground-series-s6e8"
)
blend_path = Path(
    "/kaggle/input/datasets/anthonytherrien/predicting-smartphone-addiction-vault"
)
output_path = Path("submission.csv")
seed = 42

set_seed(seed)

In [5]:
# Define the residual block
class ResidualBlock(nn.Module):
    # Define initialization
    def __init__(self, hidden_dim, dropout):
        # Call parent constructor
        super().__init__()

        # Define layers
        self.linear1 = nn.Linear(hidden_dim, hidden_dim)
        self.linear2 = nn.Linear(hidden_dim, hidden_dim)
        self.activation = nn.ReLU()
        self.dropout = nn.Dropout(dropout)
        self.batch_norm = nn.BatchNorm1d(hidden_dim)

    # Define forward pass
    def forward(self, features):
        # Save residual
        residual = features

        # Transform features
        features = self.linear1(features)
        features = self.activation(features)
        features = self.dropout(features)
        features = self.linear2(features)

        # Add residual and normalize
        features = features + residual
        features = self.batch_norm(features)
        features = self.activation(features)

        # Return features
        return features


# Define the residual classifier
class ResidualClassifier(nn.Module):
    # Define initialization
    def __init__(self, input_dim, hidden_dim, dropout):
        # Call parent constructor
        super().__init__()

        # Define model layers
        self.input_layer = nn.Linear(input_dim, hidden_dim)
        self.activation = nn.ReLU()
        self.dropout = nn.Dropout(dropout)
        self.blocks = nn.Sequential(
            ResidualBlock(hidden_dim, dropout),
            ResidualBlock(hidden_dim, dropout),
        )
        self.output_layer = nn.Linear(hidden_dim, 1)

    # Define forward pass
    def forward(self, features):
        # Transform features
        features = self.input_layer(features)
        features = self.activation(features)
        features = self.dropout(features)
        features = self.blocks(features)

        # Return one logit per row
        return self.output_layer(features).squeeze(1)

In [6]:
# Create the preprocessing pipeline
def create_preprocessor(features):
    # Detect categorical columns
    categorical_columns = features.select_dtypes(
        include=["object", "category"]
    ).columns.tolist()

    # Detect numerical columns
    numerical_columns = features.select_dtypes(
        exclude=["object", "category"]
    ).columns.tolist()

    # Define numerical preprocessing
    numerical_transformer = Pipeline(
        steps=[
            (
                "imputer",
                SimpleImputer(
                    strategy="median",
                    add_indicator=True,
                ),
            ),
            ("scaler", StandardScaler()),
        ]
    )

    # Define categorical preprocessing
    categorical_transformer = Pipeline(
        steps=[
            (
                "imputer",
                SimpleImputer(strategy="most_frequent"),
            ),
            (
                "encoder",
                OneHotEncoder(
                    handle_unknown="ignore",
                    sparse_output=False,
                ),
            ),
        ]
    )

    # Return combined preprocessing
    return ColumnTransformer(
        transformers=[
            ("num", numerical_transformer, numerical_columns),
            ("cat", categorical_transformer, categorical_columns),
        ]
    )

In [ ]:
# Prepare train, validation, and test matrices
def preprocess_data(train_df, test_df, target_column, seed):
    # Separate predictors, target, and unprocessed training ids
    features = train_df.drop(columns=["id", target_column])
    target = train_df[target_column].to_numpy(dtype=np.float32)
    train_ids = train_df["id"].copy()
    test_features = test_df.drop(columns=["id"])

    # Split before fitting preprocessing to prevent leakage
    (
        train_features,
        valid_features,
        train_target,
        valid_target,
        train_ids,
        valid_ids,
    ) = train_test_split(
        features,
        target,
        train_ids,
        test_size=0.1,
        random_state=seed,
        stratify=target,
    )

    # Fit preprocessing on the training fold only
    preprocessor = create_preprocessor(train_features)
    train_matrix = preprocessor.fit_transform(train_features)
    valid_matrix = preprocessor.transform(valid_features)
    test_matrix = preprocessor.transform(test_features)

    # Convert matrices to float32
    train_matrix = np.asarray(train_matrix, dtype=np.float32)
    valid_matrix = np.asarray(valid_matrix, dtype=np.float32)
    test_matrix = np.asarray(test_matrix, dtype=np.float32)

    # Return prepared data and unprocessed ids
    return (
        train_matrix,
        valid_matrix,
        train_target,
        valid_target,
        train_ids.reset_index(drop=True),
        valid_ids.reset_index(drop=True),
        test_matrix,
        test_df["id"].copy(),
    )

In [10]:
def create_loader(features, target=None, batch_size=4096, shuffle=False):
    # Convert features to a tensor
    feature_tensor = torch.from_numpy(features)

    # Create an inference loader
    if target is None:
        return DataLoader(
            feature_tensor,
            batch_size=batch_size,
            shuffle=False,
            pin_memory=torch.cuda.is_available(),
        )

    # Convert target to a tensor
    target_tensor = torch.from_numpy(target)

    # Create a supervised loader
    dataset = TensorDataset(feature_tensor, target_tensor)
    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        pin_memory=torch.cuda.is_available(),
    )


# Predict positive-class probabilities
def predict_probabilities(model, loader, device):
    # Set evaluation mode
    model.eval()

    # Initialize predictions
    predictions = []

    # Disable gradient tracking
    with torch.no_grad():
        # Iterate batches
        for batch in loader:
            # Support supervised and inference loaders
            batch_features = batch[0] if isinstance(batch, list) else batch
            batch_features = batch_features.to(device, non_blocking=True)

            # Convert logits to probabilities
            logits = model(batch_features)
            probabilities = torch.sigmoid(logits)
            predictions.append(probabilities.cpu().numpy())

    # Concatenate predictions
    return np.concatenate(predictions)

In [ ]:
# Add full-training leakage-safe LightGBM probability features
def add_lightgbm_features(
    train_matrix,
    valid_matrix,
    test_matrix,
    train_target,
    valid_target,
    train_ids,
    valid_ids,
    seed,
    folds=5,
    oof_output_path=Path("public_submissions/oofs/lightgbm_oof.npy"),
    test_output_path=Path("public_submissions/probs/lightgbm_test.npy"),
):
    # Combine all training partitions with their unprocessed ids
    full_train_matrix = np.vstack([train_matrix, valid_matrix])
    full_train_target = np.concatenate([train_target, valid_target])
    full_train_ids = pd.concat(
        [train_ids, valid_ids],
        ignore_index=True,
    )

    # Sort every full-training row by its original id
    sort_order = np.argsort(full_train_ids.to_numpy())
    sorted_train_matrix = full_train_matrix[sort_order]
    sorted_train_target = full_train_target[sort_order]
    sorted_train_ids = full_train_ids.to_numpy()[sort_order]

    # Ensure the saved OOF vector can be joined to ascending ids
    if not np.all(sorted_train_ids[:-1] <= sorted_train_ids[1:]):
        raise ValueError("Training ids must be sortable in ascending order.")

    # Initialize full-training OOF and test prediction features
    sorted_oof_feature = np.zeros(len(sorted_train_matrix), dtype=np.float32)
    test_lgbm_feature = np.zeros(len(test_matrix), dtype=np.float64)

    # Create stratified folds across the sorted complete training data
    splitter = StratifiedKFold(
        n_splits=folds,
        shuffle=True,
        random_state=seed,
    )

    # Train each fold only on its training subset
    for fold, (fit_indices, oof_indices) in enumerate(
        splitter.split(sorted_train_matrix, sorted_train_target),
        start=1,
    ):
        model = LGBMClassifier(
            objective="binary",
            n_estimators=1500,
            learning_rate=0.03,
            num_leaves=31,
            max_depth=-1,
            subsample=0.85,
            colsample_bytree=0.85,
            reg_alpha=0.1,
            reg_lambda=1.0,
            random_state=seed + fold,
            n_jobs=-1,
            verbosity=-1,
        )

        model.fit(
            sorted_train_matrix[fit_indices],
            sorted_train_target[fit_indices],
            eval_set=[
                (
                    sorted_train_matrix[oof_indices],
                    sorted_train_target[oof_indices],
                )
            ],
            eval_metric="auc",
            callbacks=[],
        )

        # Store predictions for this fold's held-out rows
        fold_oof_predictions = model.predict_proba(
            sorted_train_matrix[oof_indices]
        )[:, 1]
        sorted_oof_feature[oof_indices] = fold_oof_predictions

        # Average every fold model's test predictions
        test_lgbm_feature += model.predict_proba(test_matrix)[:, 1] / folds

        fold_auc = roc_auc_score(
            sorted_train_target[oof_indices],
            fold_oof_predictions,
        )
        print(f"LightGBM fold {fold:02d} AUC: {fold_auc:.6f}")

    # Report performance for the complete sorted OOF vector
    oof_auc = roc_auc_score(sorted_train_target, sorted_oof_feature)
    print(f"LightGBM full-training OOF AUC: {oof_auc:.6f}")

    # Save OOF probabilities in ascending-id order and test probabilities
    oof_output_path = Path(oof_output_path)
    test_output_path = Path(test_output_path)
    oof_output_path.parent.mkdir(parents=True, exist_ok=True)
    test_output_path.parent.mkdir(parents=True, exist_ok=True)
    np.save(oof_output_path, sorted_oof_feature)
    np.save(test_output_path, test_lgbm_feature)
    print(f"LightGBM OOF probabilities saved to {oof_output_path}")
    print(f"LightGBM test probabilities saved to {test_output_path}")

    # Map sorted OOF predictions back to original train/validation positions
    full_oof_feature = np.empty_like(sorted_oof_feature)
    full_oof_feature[sort_order] = sorted_oof_feature
    train_size = len(train_matrix)
    train_lgbm_feature = full_oof_feature[:train_size]
    valid_lgbm_feature = full_oof_feature[train_size:]

    # Append one LightGBM probability column to every matrix
    train_matrix = np.column_stack(
        [train_matrix, train_lgbm_feature]
    ).astype(np.float32)
    valid_matrix = np.column_stack(
        [valid_matrix, valid_lgbm_feature]
    ).astype(np.float32)
    test_matrix = np.column_stack(
        [test_matrix, test_lgbm_feature]
    ).astype(np.float32)

    # Return augmented feature matrices
    return train_matrix, valid_matrix, test_matrix

In [12]:
# Train the classifier
def train_model(
    model,
    train_loader,
    valid_loader,
    valid_target,
    device,
    epochs,
    learning_rate,
):
    # Define binary classification loss
    criterion = nn.BCEWithLogitsLoss()

    # Define optimizer
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=learning_rate,
        weight_decay=1e-3,
    )

    # Initialize best validation state
    best_auc = -np.inf
    best_state = None

    # Run training epochs
    for epoch in range(epochs):
        # Set training mode
        model.train()

        # Initialize loss totals
        total_loss = 0.0
        total_rows = 0

        # Iterate training batches
        for batch_features, batch_target in train_loader:
            # Move batch to device
            batch_features = batch_features.to(device, non_blocking=True)
            batch_target = batch_target.to(device, non_blocking=True)

            # Update model weights
            optimizer.zero_grad(set_to_none=True)
            logits = model(batch_features)
            loss = criterion(logits, batch_target)
            loss.backward()
            optimizer.step()

            # Accumulate row-weighted loss
            total_loss += loss.item() * batch_features.size(0)
            total_rows += batch_features.size(0)

        # Evaluate validation ROC AUC
        valid_predictions = predict_probabilities(
            model,
            valid_loader,
            device,
        )
        valid_auc = roc_auc_score(valid_target, valid_predictions)

        # Print epoch metrics
        print(
            f"Epoch {epoch + 1:02d} | "
            f"Train BCE: {total_loss / total_rows:.6f} | "
            f"Valid AUC: {valid_auc:.6f}"
        )

        # Save an independent copy of the best weights
        if valid_auc > best_auc:
            best_auc = valid_auc
            best_state = copy.deepcopy(model.state_dict())

    # Restore best weights
    model.load_state_dict(best_state)

    # Print best score
    print(f"Best validation AUC: {best_auc:.6f}")

    # Return trained model
    return model

In [13]:
# Blend predictions and save the Kaggle submission
def blend_predictions(
    test_ids,
    prediction_configs,
    target_column,
    output_path,
):
    # Initialize blend totals
    weighted_predictions = np.zeros(len(test_ids), dtype=np.float64)
    total_weight = 0.0

    # Blend every prediction source
    for prediction_name, config in prediction_configs.items():
        # Extract predictions and weight
        predictions = np.asarray(config["predictions"], dtype=np.float64)
        weight = float(config["weight"])

        # Validate prediction length
        if len(predictions) != len(test_ids):
            raise ValueError(
                f"{prediction_name} has {len(predictions)} rows; "
                f"expected {len(test_ids)}."
            )

        # Add weighted predictions
        weighted_predictions += predictions * weight
        total_weight += weight

        # Print blend information
        print(f"Blending {prediction_name} | Weight: {weight}")

    # Validate total weight
    if total_weight <= 0.0:
        raise ValueError("The total blend weight must be positive.")

    # Compute final probabilities
    final_predictions = weighted_predictions / total_weight

    # Create submission frame
    submission = pd.DataFrame(
        {
            "id": test_ids.to_numpy(),
            target_column: np.clip(final_predictions, 0.0, 1.0),
        }
    )

    # Save submission
    submission.to_csv(output_path, index=False)

    # Print confirmation
    print(f"Submission saved to {output_path}")

In [ ]:
# Load competition data
train_df = pd.read_csv(competition_path / "train.csv")
test_df = pd.read_csv(competition_path / "test.csv")

# Validate expected schemas
expected_test_columns = set(train_df.columns) - {target_column}
if set(test_df.columns) != expected_test_columns:
    raise ValueError("Train and test feature columns do not match.")

# Prepare data
(
    train_matrix,
    valid_matrix,
    train_target,
    valid_target,
    test_matrix,
    test_ids,
) = preprocess_data(
    train_df,
    test_df,
    target_column,
    seed,
)

# Add LightGBM predictions as a stacked feature
train_matrix, valid_matrix, test_matrix = add_lightgbm_features(
    train_matrix=train_matrix,
    valid_matrix=valid_matrix,
    test_matrix=test_matrix,
    train_target=train_target,
    valid_target=valid_target,
    seed=seed,
)

# Create loaders
train_loader = create_loader(
    train_matrix,
    train_target,
    batch_size=4096,
    shuffle=True,
)
valid_loader = create_loader(
    valid_matrix,
    batch_size=8192,
)
test_loader = create_loader(
    test_matrix,
    batch_size=8192,
)

# Select compute device
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)
print(f"Using device: {device}")
print(f"Processed feature count: {train_matrix.shape[1]}")

# Create classifier
model = ResidualClassifier(
    input_dim=train_matrix.shape[1],
    hidden_dim=128,
    dropout=0.2,
).to(device)

# Train classifier
model = train_model(
    model=model,
    train_loader=train_loader,
    valid_loader=valid_loader,
    valid_target=valid_target,
    device=device,
    epochs=24,
    learning_rate=1e-3,
)

# Predict test probabilities
test_predictions = predict_probabilities(
    model,
    test_loader,
    device,
)

# Load external submission files
submission_one = pd.read_csv(blend_path / "submission.csv")
submission_two = pd.read_csv(blend_path / "submission (1).csv")

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


LightGBM fold 01 AUC: 0.962454


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


LightGBM fold 02 AUC: 0.962542


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


LightGBM fold 03 AUC: 0.961648


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


LightGBM fold 04 AUC: 0.962120


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


LightGBM fold 05 AUC: 0.961774
LightGBM OOF AUC: 0.962105
LightGBM validation AUC: 0.961100
Using device: cuda
Processed feature count: 27
Epoch 01 | Train BCE: 0.308013 | Valid AUC: 0.956577
Epoch 02 | Train BCE: 0.242041 | Valid AUC: 0.959768
Epoch 03 | Train BCE: 0.232940 | Valid AUC: 0.960148
Epoch 04 | Train BCE: 0.230000 | Valid AUC: 0.960510
Epoch 05 | Train BCE: 0.228291 | Valid AUC: 0.960801
Epoch 06 | Train BCE: 0.227360 | Valid AUC: 0.960471
Epoch 07 | Train BCE: 0.226592 | Valid AUC: 0.960976
Epoch 08 | Train BCE: 0.226163 | Valid AUC: 0.960741
Epoch 09 | Train BCE: 0.225677 | Valid AUC: 0.960549
Epoch 10 | Train BCE: 0.225201 | Valid AUC: 0.960968
Epoch 11 | Train BCE: 0.225108 | Valid AUC: 0.960503
Epoch 12 | Train BCE: 0.224721 | Valid AUC: 0.960875
Epoch 13 | Train BCE: 0.224495 | Valid AUC: 0.959583
Epoch 14 | Train BCE: 0.224255 | Valid AUC: 0.959811
Epoch 15 | Train BCE: 0.223979 | Valid AUC: 0.959601
Epoch 16 | Train BCE: 0.224019 | Valid AUC: 0.960171
Epoch 17 | Tr

In [23]:
# Validate external submission ids
for submission_name, submission in {
    "sub1": submission_one,
    "sub2": submission_two,
}.items():
    if not np.array_equal(submission["id"].to_numpy(), test_ids.to_numpy()):
        raise ValueError(f"Ids in {submission_name} do not match test.csv.")

# Define prediction blend
prediction_configs = {
    "sub1": {
        "predictions": submission_one[target_column].to_numpy(),
        "weight": 2.9,
    },
    "sub2": {
        "predictions": submission_two[target_column].to_numpy(),
        "weight": 0.1,
    },
    "nn": {
        "predictions": test_predictions,
        "weight": 1e-4,
    },
}

# Blend predictions and save submission
blend_predictions(
    test_ids=test_ids,
    prediction_configs=prediction_configs,
    target_column=target_column,
    output_path=output_path,
)


Blending sub1 | Weight: 2.9
Blending sub2 | Weight: 0.1
Blending nn | Weight: 0.0001
Submission saved to submission.csv
